# CV-13.3 — FLUX.1-schnell sanity probe (Kaggle)

Generates ONE drone-style frame of T-72 tank in autumn rasputitsa landscape, pushes PNG + JSON sidecar to private dataset `Dariachup/yolo-bluebierd-data` under `_sanity/v0_tank/flux_schnell/`.

**Before running:**
1. Settings → Accelerator → **GPU T4 x2** (or just T4).
2. Settings → Internet → **On**.
3. Add-ons → Secrets → add `HF_TOKEN` with your HF write token.
4. Accept FLUX.1-schnell license at https://huggingface.co/black-forest-labs/FLUX.1-schnell

Acceptance (CV-13.3): один кадр візуально розпізнається як танк у багнюці.

In [ ]:
!pip install -q -U 'diffusers>=0.30' 'transformers>=4.44' accelerate sentencepiece protobuf 'huggingface_hub>=0.25'

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
from huggingface_hub import whoami
print('logged in as:', whoami(token=os.environ['HF_TOKEN'])['name'])

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  gpu{i}: {p.name}  vram={p.total_memory/1e9:.1f} GB')

In [ ]:
MODEL_ID = 'black-forest-labs/FLUX.1-schnell'
REPO_ID  = 'Dariachup/yolo-bluebierd-data'
REMOTE_DIR = '_sanity/v0_tank/flux_schnell'

PROMPT = (
    'oblique aerial drone reconnaissance photo, looking down at angle from above, '
    'from 400 meters altitude at 25 degrees angle above horizon. '
    'Single T-72B3 tank, stationary, on muddy dirt road. '
    'Scene: deep brown wet mud, bare leafless trees, overcast grey sky, '
    'rasputitsa, drizzle. '
    'Style: real combat footage style, low-quality compressed video frame, '
    'slight motion blur, slightly desaturated colors, EO daylight spectrum. '
    'Framing: centered single target, partially visible terrain context, '
    'no overlay text.'
)
SEED = 42
STEPS = 4
GUIDANCE = 0.0  # FLUX-schnell uses 0 guidance
MAX_SEQ = 256
HEIGHT = 512  # T4 14.5 GB не тягне 1024×1024 навіть з offload; 512 для sanity вистачить
WIDTH  = 512

In [ ]:
from diffusers import FluxPipeline
import torch

print(f'[load] {MODEL_ID}')
pipe = FluxPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
# Kaggle T4 = 14.5 GB. model_cpu_offload не влазить → потрібен sequential_cpu_offload (3-5x повільніше, але влазить).
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
print('[ok] pipeline loaded with sequential offload')

In [ ]:
print(f'[gen] steps={STEPS} guidance={GUIDANCE} seed={SEED} size={HEIGHT}x{WIDTH}')
generator = torch.Generator('cpu').manual_seed(SEED)
image = pipe(
    prompt=PROMPT,
    height=HEIGHT, width=WIDTH,
    guidance_scale=GUIDANCE,
    num_inference_steps=STEPS,
    max_sequence_length=MAX_SEQ,
    generator=generator,
).images[0]
print('[ok] generated', image.size)
image

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

ts = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_dir = Path('/kaggle/working')
stem = f'tank_rasputitsa_400m_25deg_{ts}'
img_path = out_dir / f'{stem}.png'
meta_path = out_dir / f'{stem}.json'
image.save(img_path)

meta = {
    'model': MODEL_ID,
    'prompt': PROMPT,
    'steps': STEPS,
    'guidance_scale': GUIDANCE,
    'seed': SEED,
    'max_sequence_length': MAX_SEQ,
    'height': HEIGHT,
    'width': WIDTH,
    'timestamp_utc': ts,
    'image_size': list(image.size),
    'class_hint': 'tank',
    'altitude_m_hint': 400,
    'view_angle_deg_hint': 25,
    'season_hint': 'autumn_mud',
    'landscape_hint': 'dirt_mud_road',
    'source': 'kaggle_notebook',
}
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2))
print(f'[save] {img_path}  bytes={img_path.stat().st_size}')

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
for p in (img_path, meta_path):
    api.upload_file(
        path_or_fileobj=str(p),
        path_in_repo=f'{REMOTE_DIR}/{p.name}',
        repo_id=REPO_ID,
        repo_type='dataset',
        commit_message=f'sanity: flux-schnell {stem} (kaggle)',
    )
    print(f'[push] -> {REPO_ID}/{REMOTE_DIR}/{p.name}')
print('[done] check https://huggingface.co/datasets/' + REPO_ID + '/tree/main/' + REMOTE_DIR)